In [4]:
import sys
sys.path.append("..")

from src.preprocess import preprocess
from src.bm25 import BM25
from src.eval import evaluate

In [5]:
toy_docs = [["folate", "level", "homocysteine", "folate"],
            ["vitamin", "diet", "level"],
            ["tau", "protein", "neuron", "level", "brain", "tau"]]

toy = BM25(toy_docs, ["d0", "d1", "d2"])
print(toy.search(["folate", "level"], k=3))

[('d0', 1.5750270090604788), ('d1', 0.15499179501060653), ('d2', 0.1138300396143471)]


In [6]:
# Test 1: relevant doc "a" is ranked 2nd
print(evaluate({"q1": ["x", "a", "y"]}, {"q1": {"a": 1}}, k=3))
# expected: Recall@3 = 1.0, MRR@3 = 0.5, nDCG@3 = 0.6309

# Test 2: q1 found at rank 1, q2 not found
print(evaluate({"q1": ["a", "x", "y"], "q2": ["x", "y", "z"]},
               {"q1": {"a": 1}, "q2": {"b": 1}}, k=3))
# expected: all three = 0.5

{'Recall@3': 1.0, 'MRR@3': 0.5, 'nDCG@3': 0.6309}
{'Recall@3': 0.5, 'MRR@3': 0.5, 'nDCG@3': 0.5}


In [7]:
from beir.datasets.data_loader import GenericDataLoader

corpus, queries, qrels = GenericDataLoader("../data/scifact").load(split="test")
docs = {did: (d["title"] + ". " + d["text"]).strip() for did, d in corpus.items()}
doc_ids = list(docs)
tokenized = [preprocess(docs[d]) for d in doc_ids]
print(len(docs), len(queries))

100%|██████████| 5183/5183 [00:00<00:00, 114967.44it/s]


5183 300


In [8]:
bm25 = BM25(tokenized, doc_ids)

qid = list(queries)[0]
print("QUERY:", queries[qid])
relevant = set(qrels[qid])
for rank, (did, s) in enumerate(bm25.search(preprocess(queries[qid]), k=5), 1):
    mark = "✔" if did in relevant else " "
    print(f"{rank}. {mark} {s:.2f} | {docs[did][:90]}")

QUERY: 0-dimensional biomaterials show inductive properties.
1.   10.47 | High-performance neuroprosthetic control by an individual with tetraplegia.. BACKGROUND Pa
2.   10.44 | The Epithelial-Mesenchymal Transition Generates Cells with Properties of Stem Cells. The e
3.   10.19 | How long will long-term potentiation last?. The paramount feature of long-term potentiatio
4.   10.03 | The first gene of the Bacillus subtilis clpC operon, ctsR, encodes a negative regulator of
5.   9.73 | Epithelial and mesenchymal subpopulations within normal basal breast cell lines exhibit di


In [9]:
bm25_results = {}
for qid, q in queries.items():
    bm25_results[qid] = [d for d, _ in bm25.search(preprocess(q), k=10)]

print("BM25  :", evaluate(bm25_results, qrels))

BM25  : {'Recall@10': 0.8302, 'MRR@10': 0.6527, 'nDCG@10': 0.6903}


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(analyzer=preprocess)
X = vec.fit_transform([docs[d] for d in doc_ids])

tfidf_results = {}
for qid, q in queries.items():
    sims = (X @ vec.transform([q]).T).toarray().ravel()
    tfidf_results[qid] = [doc_ids[i] for i in sims.argsort()[::-1][:10]]

print("TF-IDF:", evaluate(tfidf_results, qrels))

TF-IDF: {'Recall@10': 0.7542, 'MRR@10': 0.5616, 'nDCG@10': 0.5992}


In [12]:
import pandas as pd, os

table = pd.DataFrame({
    "TF-IDF": evaluate(tfidf_results, qrels),
    "BM25":   evaluate(bm25_results, qrels),
}).T
print(table)

os.makedirs("../results", exist_ok=True)
table.to_csv("../results/day2_keyword_metrics.csv")

        Recall@10  MRR@10  nDCG@10
TF-IDF     0.7542  0.5616   0.5992
BM25       0.8302  0.6527   0.6903


In [13]:
def first_hit(results, qid):
    for i, d in enumerate(results[qid]):
        if d in qrels[qid]:
            return i + 1
    return None                      # relevant doc not in the top 10

bm25_only, tfidf_only, both_miss = [], [], []
for qid in queries:
    b, t = first_hit(bm25_results, qid), first_hit(tfidf_results, qid)
    if b and not t:
        bm25_only.append(qid)
    elif t and not b:
        tfidf_only.append(qid)
    elif not b and not t:
        both_miss.append(qid)

print("Only BM25 found it:  ", len(bm25_only))
print("Only TF-IDF found it:", len(tfidf_only))
print("Both missed it:      ", len(both_miss))

Only BM25 found it:   29
Only TF-IDF found it: 7
Both missed it:       38


In [14]:
def show(qid):
    print("QUERY:", queries[qid])
    for did in qrels[qid]:
        print("RELEVANT:", docs[did][:300], "\n")

if bm25_only:
    print("--- BM25 found, TF-IDF missed ---"); show(bm25_only[0])
if tfidf_only:
    print("--- TF-IDF found, BM25 missed ---"); show(tfidf_only[0])
if both_miss:
    print("--- Both missed ---"); show(both_miss[0])

--- BM25 found, TF-IDF missed ---
QUERY: Autophagy declines in aged organisms.
RELEVANT: Restoration of chaperone-mediated autophagy in aging liver improves cellular maintenance and hepatic function. Chaperone-mediated autophagy (CMA), a selective mechanism for degradation of cytosolic proteins in lysosomes, contributes to the removal of altered proteins as part of the cellular quality- 

--- TF-IDF found, BM25 missed ---
QUERY: Cellular aging closely links to an older appearance.
RELEVANT: Perceived age as clinically useful biomarker of ageing: cohort study.. OBJECTIVE To determine whether perceived age correlates with survival and important age related phenotypes. DESIGN Follow-up study, with survival of twins determined up to January 2008, by which time 675 (37%) had died. SETTING P 

--- Both missed ---
QUERY: 0-dimensional biomaterials show inductive properties.
RELEVANT: New opportunities: the use of nanotechnologies to manipulate and track stem cells.. Nanotechnologies are emerg